### Polynomial Regression with Gradient Descent

In this implementation, we model the relationship between engine size and CO2 emissions using this function:

$$
\hat{y} = ax^2 + bx + c
$$

Where:
- $x$: Engine size
- $\hat{y}$: Predicted CO2 emission
- $a, b, c$: Model parameters to be optimized via Gradient Descent


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Data Loading and Preprocessing
We load the dataset and extract the relevant features:
- **X**: Engine Size (Independent variable)
- **Y**: CO2 Emissions (Dependent variable)


In [ ]:
# X= Engin_Size, Y= CO2EMISSIONS
data_test = np.genfromtxt("FuelConsumption.csv", delimiter=",", dtype=str)[1:, :] #.astype(float)
X , Y = data_test[:,(4,12)].astype(float).T

# Z = ax+by+c

In [ ]:
a, b, c = np.random.random(3)
learning_rate = 0.001
iterations = 100
data_length = X.shape[0]

for i in range(iterations):
    Z = a * X + b * Y + c

    da = (2 / data_length) * np.sum((Z - Y) * X)
    db = (2 / data_length) * np.sum((Z - Y) * Y)
    dc = (2 / data_length) * np.sum(Z - Y)

    a = a - learning_rate * da
    b = b - learning_rate * db
    c = c - learning_rate * dc



# Scale Z manually to avoid overflow
z_scale = 1e213
Zs = Z / z_scale

P = np.column_stack((X, Y, Zs))

# Normalize
mu = P.mean(axis=0)
sigma = P.std(axis=0)
sigma[sigma == 0] = 1.0

Pn = (P - mu) / sigma
n = Pn.shape[0]

# line: p0 + t*v
p0 = Pn.mean(axis=0).copy()
v = np.array([1.0, 1.0, 1.0], dtype=np.float64)
v /= np.linalg.norm(v)

t = (Pn - p0) @ v

lr = 0.01
epochs = 5000

for _ in range(epochs):
    P_pred = p0 + np.outer(t, v)
    E = P_pred - Pn

    grad_p0 = (2.0 / n) * np.sum(E, axis=0)
    grad_v = (2.0 / n) * np.sum(E * t[:, None], axis=0)
    grad_t = (2.0 / n) * np.sum(E * v[None, :], axis=1)

    p0 -= lr * grad_p0
    v -= lr * grad_v
    t -= lr * grad_t

    nv = np.linalg.norm(v)
    if nv < 1e-12:
        break
    v /= nv

# Build a visible line segment
t_line = np.linspace(np.percentile(t, 1), np.percentile(t, 99), 200)
P_line_norm = p0 + np.outer(t_line, v)

# Back to original scale
P_line_scaled = mu + sigma * P_line_norm
P_line = P_line_scaled.copy()
P_line[:, 2] *= z_scale

# Safety check
print("finite line:", np.isfinite(P_line).all())
print("line start:", P_line[0])
print("line end:", P_line[-1])

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(X, Y, Z, c="blue", s=12, alpha=0.35)
ax.plot(P_line[:, 0], P_line[:, 1], P_line[:, 2], c="red", lw=4)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D Fitted Line")

ax.set_xlim(min(X.min(), P_line[:, 0].min()), max(X.max(), P_line[:, 0].max()))
ax.set_ylim(min(Y.min(), P_line[:, 1].min()), max(Y.max(), P_line[:, 1].max()))
ax.set_zlim(min(Z.min(), P_line[:, 2].min()), max(Z.max(), P_line[:, 2].max()))

plt.savefig('plot_comparison.png', dpi=300)

plt.tight_layout()
plt.show()
